# Attempt 2 with Langchain only

In [1]:
from os import environ
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings_client = OpenAIEmbeddings(
    base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model=environ.get("GITHUB_EMBEDDINGS_MODEL_ID")  # 🎯 Selected AI model
)

In [3]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="LLM_Powered_Autonomous_Agents",
    embedding_function=embeddings_client,
    persist_directory="./.chroma_db"
)

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
# A special class/object that filters HTML documents for relevant data
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
# A "Loader" is an object representing the method for grabbing data, in this case, from a public website with sample data
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [5]:
# Print out first 500 characters from the document
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [6]:
from langchain_experimental.text_splitter import SemanticChunker

text_splitter = SemanticChunker(
    embeddings=embeddings_client,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 53 sub-documents.


In [7]:
# TODO: Find a semantic method of splitting a document via a "change in context", i.e. a topic change

# from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,  # chunk size (characters)
#     chunk_overlap=200,  # chunk overlap (characters)
#     add_start_index=True,  # track index in original document
# )
# all_splits = text_splitter.split_documents(docs)

# print(f"Split blog post into {len(all_splits)} sub-documents.")

In [8]:
# Shoves our document chunks into the embeddings model, and stores them in our local chroma vector store database
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['457bc36e-61db-4a48-abe1-009ae5f38d4e', '3a74b2da-ee45-40ea-96a9-186e2b090696', 'e539ed2d-5bef-4835-86ed-b4dad03b8917']


In [9]:
# Defines a simple RAG tool function for an agent to use
from langchain.tools import tool

# The "response_format=" argument is used when a tool function returns 2 values:
# 1. The string message result to send to the model
# 2. An "artifact" to couple with the tool's result, in this case, the actual retrieved document objects themselves
# In this case, this allows us (not necessarily the LLM in the agent) to access the document's metadata more easily when we interact with the RAG tool
# @tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""

    # K-nearest neighbours search with Chroma
    retrieved_docs = vector_store.similarity_search(query, k=2)

    # The "serialised" result, which is just a long string combining documents (chunks) with their metadata
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )

    print(serialized)

    return serialized, retrieved_docs

# Extra note: you can easily ask the calling LLM to provide more arguments to a tool function:
# from typing import Literal
# def retrieve_context(query: str, section: Literal["beginning", "middle", "end"]):

In [10]:
from langchain_openai import ChatOpenAI

chat_client = ChatOpenAI(
    base_url= environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key= environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model= environ.get("GITHUB_MODEL_ID")  # 🎯 Selected AI model
)

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the prompt structure
prompt = ChatPromptTemplate.from_messages([
    ("system", "You have access to a tool that retrieves context from a blog post. Use the tool to help answer user queries."),
    ("user", "{user_input}"),
    ("assistant", "Context:\n\n{context_input}")
])

# 2. Create a simple chain by piping the prompt to the model
llm_chain = prompt | chat_client

# The query from your notebook
query = "What is the standard method for Task Decomposition?"

# 3. Invoke the chain with the input variable
response = llm_chain.invoke({"user_input": query, "context_input": retrieve_context(query)})

print(response.content)

Source: {'start_index': 2578, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable plann